# Hugging Face Transformers — Əsas Tətbiqlər

Bu notebook `transformers` kitabxanasının ən çox istifadə olunan tapşırıqlarını göstərir: mətn təsnifatı, NER, sual-cavab, mətn generasiyası, xülasə, tərcümə və öz datanla fine-tuning.

## `pipeline` nədir?

`pipeline` — Hugging Face-in ən sadə API-sidir. Tokenizasiya, modelin işə salınması və nəticənin emalı kimi bütün ara addımları gizlədir, sən yalnız tapşırıq adı və (istəsən) model adı verirsən, qalanını o edir.

```python
pipeline("tapşırıq_adı", model="model_adı")  # model adı verilməsə default model yüklənir
```

Produksiyada `AutoTokenizer` + `AutoModel` ilə daha çox nəzarət (batch processing, custom loss və s.) əldə edilir, amma öyrənmək və prototip üçün `pipeline` kifayətdir.

> **Colab GPU:** `Runtime → Change runtime type → GPU (T4)`. Kiçik modellər CPU-da da işləyir, amma GPU generasiya tapşırıqlarını xeyli sürətləndirir.

In [ ]:
!pip install -q transformers torch

In [ ]:
from transformers import pipeline

## 1. Sentiment analiz (mətn təsnifatı)
Mətnin emosional tonunu (müsbət/mənfi) təyin edir.

In [ ]:
classifier = pipeline("sentiment-analysis")
result = classifier("I love this product, it works great!")
print(result)

## 2. Named Entity Recognition (NER)
Mətndəki şəxs, təşkilat, yer adları kimi varlıqları tapır.

In [ ]:
ner = pipeline("ner", grouped_entities=True)
result = ner("PASHA Bank is headquartered in Baku, Azerbaijan.")
print(result)

## 3. Sual-cavab (question answering)
Verilmiş kontekst daxilində suala cavab tapır (kontekstdən kənar məlumat uydurmur).

In [ ]:
qa = pipeline("question-answering")

context = "Model risk management involves validating quantitative models used in banking."
question = "What does model risk management involve?"

print(qa(question=question, context=context))

## 4. Mətn generasiyası
Verilmiş başlanğıc mətni davam etdirir.

In [ ]:
generator = pipeline("text-generation", model="gpt2")
result = generator("The future of AI in finance is", max_length=50, num_return_sequences=1)
print(result[0]["generated_text"])

## 5. Xülasə (summarization)
Uzun mətni qısa xülasəyə çevirir.

In [ ]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

text = """
Artificial intelligence is transforming industries across the world, from healthcare to finance.
Machine learning models can now analyze vast amounts of data to identify patterns humans might miss.
However, this rapid adoption also raises questions about ethics, bias, and regulation that
companies and governments are still working to address.
"""

print(summarizer(text, max_length=60, min_length=20, do_sample=False))

## 6. Tərcümə
Bir dildən digərinə tərcümə edir.

In [ ]:
translator = pipeline("translation_en_to_fr", model="Helsinki-NLP/opus-mt-en-fr")
print(translator("Hello, how are you?"))

## 7. Öz datanla fine-tuning (əsas skelet)

`pipeline` hazır modellərlə işləyir, amma öz datana uyğunlaşdırmaq istəsən modeli fine-tune etmək lazımdır. Aşağıda skelet var — `dataset` dəyişənini öz datanla (məs. `datasets` kitabxanasından yüklənmiş dataset) əvəz etməlisən.

In [ ]:
!pip install -q datasets

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# nümunə: hazır kiçik dataset (öz datanla əvəz et)
dataset = load_dataset("imdb")
small_train = dataset["train"].shuffle(seed=42).select(range(500))  # sürətli test üçün kiçildilib
small_test = dataset["test"].shuffle(seed=42).select(range(200))

def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True)

small_train = small_train.map(tokenize, batched=True)
small_test = small_test.map(tokenize, batched=True)

In [ ]:
args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=small_train,
    eval_dataset=small_test,
)

trainer.train()

In [ ]:
# Fine-tune olunmuş modellə test
finetuned_classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)
finetuned_classifier("This movie was absolutely wonderful!")

## Növbəti addımlar

- Hər tapşırıq üçün `model=` parametri ilə [Model Hub](https://huggingface.co/models)-dan daha uyğun/güclü model seç — konkret model performansı əhəmiyyətli dərəcədə dəyişir
- Fine-tuning-də `num_train_epochs`-u artır və tam datasetlə işlə (yuxarıda sürət üçün kiçildilib)
- Batch processing və GPU idarəetməsi üçün `pipeline` əvəzinə birbaşa `AutoTokenizer` + `AutoModel` istifadə et
- LoRA / PEFT ilə böyük modelləri az resursla fine-tune etməyi araşdır